In [1]:
!pip install pandas numpy gensim scikit-learn xgboost nltk

  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 2.8 MB/s eta 0:00:04
   ----- ---------------------------------- 1.6/11.3 MB 3.8 MB/s eta 0:00:03
   ---------- ----------------------------- 2.9/11.3 MB 4.9 MB/s eta 0:00:02
   ----------------- ---------------------- 5.0/11.3 MB 6.2 MB/s eta 0:00:02
   -------------------------- ------------- 7.6/11.3 MB 7.5 MB/s eta 0:00:01
   ------------------------------------ --- 10.2/11.3 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 8.6 MB/s  0:00:01
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------- ----------------------------- 3.4/12.9 MB 16.8 MB/s eta 0:00:01
   --------------------- ------------------ 7.1/12

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lalit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lalit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [2]:
import pandas as pd

# complaints dataset
df = pd.read_csv("complaints.csv")

# products dataset
products = pd.read_csv("products.csv")

In [3]:
import re
from nltk.tokenize import word_tokenize

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

df['clean_text'] = df['ReturnMessage'].apply(clean_text)
df['tokens'] = df['clean_text'].apply(word_tokenize)

In [4]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

w2v_model.save("w2v.model")

In [5]:
import numpy as np

def sentence_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

df['embedding'] = df['tokens'].apply(lambda x: sentence_vector(x, w2v_model))

In [6]:
fraud_vectors = {}

for fraud_type in df['Fraud_Type'].unique():
    subset = df[df['Fraud_Type'] == fraud_type]
    vectors = np.vstack(subset['embedding'].values)
    fraud_vectors[fraud_type] = np.mean(vectors, axis=0)

In [7]:
colors = ['red', 'blue', 'black', 'white', 'green', 'purple']
sizes = ['small', 'medium', 'large', 'xl', 'xxl']

def extract_attribute(text, attr_list):
    for attr in attr_list:
        if attr in text:
            return attr
    return None

In [8]:
def contradiction_score(row):
    text = row['clean_text']
    product = products[products['ProductID'] == row['ProductID']].iloc[0]['ProductName'].lower()
    
    claimed_color = extract_attribute(text, colors)
    actual_color = extract_attribute(product, colors)
    
    if claimed_color and actual_color and claimed_color != actual_color:
        return 1.0
    
    return 0.0

In [9]:
def behavioral_score(text):
    score = 0
    
    if text.isupper():
        score += 0.3
        
    if "refund" in text:
        score += 0.2
        
    if "wore" in text or "used" in text:
        score += 0.5
        
    return min(score, 1.0)

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_features(vec):
    scores = {}
    for k, v in fraud_vectors.items():
        scores[k] = cosine_similarity([vec], [v])[0][0]
    return scores

In [11]:
feature_rows = []

for _, row in df.iterrows():
    vec = row['embedding']
    
    sem_scores = semantic_features(vec)
    
    contra = contradiction_score(row)
    behavior = behavioral_score(row['clean_text'])
    
    feature = {
        'semantic_contradiction': sem_scores.get('Contradiction_Lie', 0),
        'semantic_switcheroo': sem_scores.get('Switcheroo_Scam', 0),
        'semantic_wardrobe': sem_scores.get('Wardrobing_Abuse', 0),
        'contradiction_score': contra,
        'behavior_score': behavior
    }
    
    feature_rows.append(feature)

X = pd.DataFrame(feature_rows)
y = df['Fraud_Prob_%']

In [12]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1
)

model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [13]:
from sklearn.metrics import mean_squared_error

preds = model.predict(X_test)
print("MSE:", mean_squared_error(y_test, preds))

MSE: 769.8888752738545


In [14]:
def predict_fraud(text, product_id):
    clean = clean_text(text)
    tokens = word_tokenize(clean)
    
    vec = sentence_vector(tokens, w2v_model)
    sem_scores = semantic_features(vec)
    
    product = products[products['ProductID'] == product_id].iloc[0]
    
    row = {
        'clean_text': clean,
        'ProductID': product_id
    }
    
    contra = contradiction_score(row)
    behavior = behavioral_score(clean)
    
    features = [[
        sem_scores.get('Contradiction_Lie', 0),
        sem_scores.get('Switcheroo_Scam', 0),
        sem_scores.get('Wardrobing_Abuse', 0),
        contra,
        behavior
    ]]
    
    score = model.predict(features)[0]
    
    return score

In [15]:
text = "I want to change the color, cause i want to refund"
product_id = "P-101"

print(predict_fraud(text, product_id))

85.38821


In [17]:
text = "The color arrived wrong, the strap was broken"
product_id = "P-101"

print(predict_fraud(text, product_id))

27.219687


In [5]:
import pandas as pd
from datetime import datetime, timedelta

# -----------------------------
# CONFIG
# -----------------------------
NUM_ORDERS = 800
NUM_CUSTOMERS = 100
START_PRODUCT_ID = 201
NUM_PRODUCTS = 100
START_DATE = datetime(2026, 2, 1)

# -----------------------------
# LOAD PRODUCTS
# -----------------------------
products = pd.read_csv("Products.csv")

# Validate required columns
required_cols = {"product_id", "product_name", "color"}
if not required_cols.issubset(products.columns):
    raise ValueError(f"products.csv must contain columns: {required_cols}")

# Create lookup
product_lookup = {
    row["product_id"]: {
        "name": row["product_name"],
        "color": row["color"]
    }
    for _, row in products.iterrows()
}

# -----------------------------
# GENERATE ORDERS
# -----------------------------
orders = []

for i in range(1, NUM_ORDERS + 1):
    
    # Proper formatting (O-001 → O-800)
    order_id = f"O-{i:03}"
    
    # Customers: C-001 → C-100
    customer_id = f"C-{((i - 1) % NUM_CUSTOMERS) + 1:03}"
    
    # Products: P-201 → P-300
    product_id_num = START_PRODUCT_ID + ((i - 1) % NUM_PRODUCTS)
    product_id = f"P-{product_id_num}"
    
    # Safety check
    if product_id not in product_lookup:
        raise ValueError(f"Product ID {product_id} not found in products.csv")
    
    product_info = product_lookup[product_id]
    
    # Spread dates across 60 days
    purchase_date = START_DATE + timedelta(days=(i % 60))
    
    orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "product_id": product_id,
        "product_name": product_info["name"],
        "product_category": "Electronics",
        "product_color": product_info["color"],
        "purchase_date": purchase_date.strftime("%Y-%m-%d")
    })

# -----------------------------
# SAVE CSV
# -----------------------------
df = pd.DataFrame(orders)

# Final sanity check
assert len(df) == NUM_ORDERS

df.to_csv("orders.csv", index=False)

print("✅ orders.csv generated successfully with", len(df), "rows")

✅ orders.csv generated successfully with 800 rows


In [6]:
import pandas as pd
import sqlite3

# -----------------------------
# FILE PATHS
# -----------------------------
CUSTOMERS_FILE = "Customers.csv"
PRODUCTS_FILE = "Products.csv"
ORDERS_FILE = "Orders.csv"
COMPLAINTS_FILE = "Complaints.csv"

DB_FILE = "ecommerce.db"

# -----------------------------
# LOAD CSVs
# -----------------------------
customers = pd.read_csv(CUSTOMERS_FILE)
products = pd.read_csv(PRODUCTS_FILE)
orders = pd.read_csv(ORDERS_FILE)
complaints = pd.read_csv(COMPLAINTS_FILE)

# -----------------------------
# CREATE DATABASE
# -----------------------------
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# -----------------------------
# WRITE TABLES
# -----------------------------
customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
complaints.to_sql("complaints", conn, if_exists="replace", index=False)

# -----------------------------
# ADD INDEXES (IMPORTANT 🔥)
# -----------------------------
cursor.execute("CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_orders_product ON orders(product_id);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_complaints_order ON complaints(order_id);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_complaints_customer ON complaints(customer_id);")

conn.commit()

print("✅ Database created successfully: ecommerce.db")

# -----------------------------
# TEST QUERY
# -----------------------------
query = """
SELECT c.complaint_text, c.fraud_classification, o.product_id
FROM complaints c
JOIN orders o ON c.order_id = o.order_id
LIMIT 5;
"""

df = pd.read_sql(query, conn)
print("\nSample Join Output:")
print(df)

conn.close()

✅ Database created successfully: ecommerce.db

Sample Join Output:
                                      complaint_text fraud_classification  \
0  My Sony headphones stopped working completely ...           Legitimate   
1  Laptop screen is completely black not turning ...           Legitimate   
2  Battery exploded while charging this is danger...           Legitimate   
3           Phone is dead on arrival not powering on           Legitimate   
4       Earbuds only work on one side defective unit           Legitimate   

  product_id  
0      P-201  
1      P-202  
2      P-203  
3      P-204  
4      P-205  
